# Claude API Fundamentals — AI Engineering Interview Prep

Completions, streaming, system prompts, token counting, conversation history.

**Prerequisites**: Set `ANTHROPIC_API_KEY` environment variable before running.

In [ ]:
import anthropic
import os
import json
import time

# Initialize client
client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY")
)

MODEL = "claude-haiku-4-5-20251001"
print(f"Using model: {MODEL}")
print(f"API key set: {'Yes' if os.environ.get('ANTHROPIC_API_KEY') else 'No — set ANTHROPIC_API_KEY'}")

## 1. Basic Completion

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "What is the difference between precision and recall in ML?"}
    ]
)

print("Response:")
print(message.content[0].text)
print(f"\nUsage: input={message.usage.input_tokens} tokens, output={message.usage.output_tokens} tokens")
print(f"Stop reason: {message.stop_reason}")

## 2. System Prompt

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system="""You are a concise ML interviewer. When asked a concept question:
1. Define it in one sentence
2. Give one concrete example
3. State when to use it vs alternatives
Keep each answer under 150 words.""",
    messages=[
        {"role": "user", "content": "Explain gradient boosting."}
    ]
)

print(message.content[0].text)

## 3. Streaming

In [ ]:
print("Streaming response:")
print("-" * 40)

full_text = ""
start = time.perf_counter()

with client.messages.stream(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": "List 5 key differences between SQL and NoSQL databases."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_text += text

elapsed = time.perf_counter() - start
print(f"\n\nTotal chars: {len(full_text)}, Time: {elapsed:.2f}s")

## 4. Multi-Turn Conversation

In [ ]:
def chat(messages, system=None):
    """Send a conversation and return the assistant's reply."""
    kwargs = {"model": MODEL, "max_tokens": 300, "messages": messages}
    if system:
        kwargs["system"] = system
    resp = client.messages.create(**kwargs)
    return resp.content[0].text

# Simulated conversation
messages = []
questions = [
    "What is overfitting?",
    "How do we detect it?",
    "What regularization techniques fix it?"
]

system = "You are a concise ML tutor. Keep answers under 80 words."

for q in questions:
    messages.append({"role": "user", "content": q})
    reply = chat(messages, system=system)
    messages.append({"role": "assistant", "content": reply})
    print(f"Q: {q}")
    print(f"A: {reply}")
    print("-" * 50)

## 5. Structured JSON Output

In [ ]:
prompt = """Evaluate this ML model description and return a JSON object with these fields:
- "model_type": string
- "strengths": list of strings (max 3)
- "weaknesses": list of strings (max 3)
- "use_cases": list of strings (max 3)
- "complexity": "low" | "medium" | "high"

Model: Random Forest — an ensemble of decision trees trained on bootstrap samples.

Return ONLY valid JSON, no other text."""

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)

raw = response.content[0].text.strip()
try:
    parsed = json.loads(raw)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("Raw response (parse failed):", raw)

## 6. Token Counting & Cost Estimation

In [ ]:
# Count tokens before sending
messages_to_count = [
    {"role": "user", "content": "Explain the transformer architecture in detail, including all components."}
]

count = client.messages.count_tokens(
    model=MODEL,
    messages=messages_to_count
)
print(f"Input tokens: {count.input_tokens}")

# Cost estimation (Haiku pricing, approximate)
HAIKU_INPUT_PER_1M  = 0.80   # $/1M tokens
HAIKU_OUTPUT_PER_1M = 4.00   # $/1M tokens

# Simulate a workload
n_requests = 1000
avg_input   = 500
avg_output  = 200

total_cost = (n_requests * avg_input / 1e6 * HAIKU_INPUT_PER_1M +
              n_requests * avg_output / 1e6 * HAIKU_OUTPUT_PER_1M)
print(f"\nEstimated cost for {n_requests} requests:")
print(f"  Avg input:  {avg_input} tokens")
print(f"  Avg output: {avg_output} tokens")
print(f"  Total: ~${total_cost:.4f}")

## Interview Tips

- **Temperature**: 0=deterministic, 1=default creativity, >1=more random. For structured output: use 0.
- **max_tokens**: hard limit on output length. Set deliberately — billing is per token.
- **stop_reason**: 'end_turn' = complete; 'max_tokens' = truncated; 'stop_sequence' = hit stop string.
- **System prompt**: sets persistent context/persona. Not shown in messages array.
- **Streaming**: reduces time-to-first-token (TTFT). Users perceive faster responses.
- **Token counting**: ~4 chars/token for English. Always count before building cost estimates.